[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/02.%20Clase%202/02Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F02.%20Clase%202%2F02Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 2: Retornos logarítmicos y matriz de covarianza

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

---

## Introducción teórica

En la Clase 1 introdujimos los rendimientos simples y logarítmicos de activos individuales. En esta clase profundizamos en las **relaciones entre múltiples activos**, herramienta fundamental para la construcción de portafolios.

### Objetivos
1. Calcular rendimientos logarítmicos de un conjunto de activos.
2. Estimar la **matriz de covarianza** y la **matriz de correlación**.
3. Visualizar las relaciones bivariadas entre rendimientos mediante gráficos de dispersión, distribuciones conjuntas y regresiones lineales.
4. Interpretar la **volatilidad móvil** y la **correlación rodante** como medidas dinámicas de riesgo y co-movimiento.

### Marco teórico: de rendimientos individuales a portafolios

Para un vector de $n$ activos con rendimientos logarítmicos $\mathbf{r}_t = (r_{1,t}, \ldots, r_{n,t})^\top$, los dos objetos estadísticos centrales son:

$$
\boldsymbol{\mu} = \mathbb{E}[\mathbf{r}_t], \qquad \boldsymbol{\Sigma} = \text{Cov}(\mathbf{r}_t) = \mathbb{E}[(\mathbf{r}_t - \boldsymbol{\mu})(\mathbf{r}_t - \boldsymbol{\mu})^\top]
$$

donde $\boldsymbol{\mu}$ es el vector de rendimientos esperados y $\boldsymbol{\Sigma}$ es la **matriz de covarianza** ($n \times n$, simétrica y semidefinida positiva).

La varianza de un portafolio con pesos $\mathbf{w}$ es:

$$
\sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \, \mathbf{w}
$$

Por lo tanto, estimar $\boldsymbol{\Sigma}$ con precisión es **crítico** para la optimización de portafolios (Markowitz, 1952).

## 1. Descarga de datos con `yfinance`

Usamos `yfinance` para descargar precios ajustados de cierre de Yahoo Finance.  
Consideramos tres acciones (Alcoa, Apple, Microsoft) y el índice S&P 500.

> **Nota BMV**: Para acciones de la Bolsa Mexicana de Valores, el ticker debe incluir la extensión `.MX`.  
> Ejemplo: `MEXCHEM.MX`, `LABB.MX`, `GFINBURO.MX`, `GFNORTEO.MX`.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Estilo visual
sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 8)

In [ ]:
tickers    = ["AA", "AAPL", "MSFT", "^GSPC"]
start_date = "2025-01-01"
end_date   = "2025-03-27"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)

closes = data["Close"]
print(f"Activos: {list(closes.columns)}  |  Observaciones: {len(closes)}")
closes.head()

---
## 2. Rendimientos logarítmicos

Recordemos que el rendimiento logarítmico se define como:

$$
r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right)
$$

y tiene la propiedad de ser **aditivo en el tiempo**: $r_{t_0 \to t_n} = \sum_{i=1}^{n} r_{t_{i-1} \to t_i}$.

Para un análisis multivariado, calculamos los rendimientos de **todos los activos simultáneamente** y obtenemos una matriz donde cada columna es la serie de rendimientos de un activo.

In [ ]:
r = np.log(closes / closes.shift(1)).dropna()
r.head()

In [ ]:
r.describe()

In [ ]:
pd.DataFrame({
    "Asimetría": r.skew(),
    "Curtosis (exceso)": r.kurtosis()
})

---
## 3. Matriz de covarianza y correlación

### 3.1 Matriz de covarianza muestral

La matriz de covarianza muestral se estima como:

$$
\hat{\boldsymbol{\Sigma}} = \frac{1}{T-1} \sum_{t=1}^{T} (\mathbf{r}_t - \bar{\mathbf{r}})(\mathbf{r}_t - \bar{\mathbf{r}})^\top
$$

El elemento $(i,j)$ es la covarianza entre los activos $i$ y $j$. La diagonal contiene las varianzas individuales.

### 3.2 Matriz de correlación

La correlación normaliza la covarianza por las desviaciones estándar:

$$
\rho_{ij} = \frac{\sigma_{ij}}{\sigma_i \, \sigma_j}, \qquad -1 \leq \rho_{ij} \leq 1
$$

**Interpretación**:
- $\rho \approx 1$: los activos se mueven juntos (poca diversificación).
- $\rho \approx 0$: movimientos independientes (buena diversificación).
- $\rho \approx -1$: movimientos opuestos (cobertura natural).

In [ ]:
cov_matrix = r.cov()
print("Matriz de covarianza muestral:")
cov_matrix

In [ ]:
# Covarianza anualizada (× 252 días hábiles)
cov_annual = cov_matrix * 252
print("Matriz de covarianza anualizada:")
cov_annual

In [ ]:
corr_matrix = r.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Matriz de correlación de rendimientos logarítmicos")
plt.tight_layout()

---
## 4. Volatilidad móvil

La **volatilidad rodante** (rolling volatility) estima la desviación estándar en una ventana móvil de $n$ días y la anualiza multiplicando por $\sqrt{252}$:

$$
\hat{\sigma}_{t}^{\text{anual}}(n) = \sqrt{252} \cdot \sqrt{\frac{1}{n-1}\sum_{i=0}^{n-1}(r_{t-i} - \bar{r}_t)^2}
$$

Esto permite observar **cómo cambia el riesgo** a lo largo del tiempo. Períodos de alta volatilidad tienden a agruparse — el fenómeno de **clustering de volatilidad** (Engle, 1982).

In [ ]:
window = 20
vol = r.rolling(window).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(12, 6))
vol.plot(ax=ax, alpha=0.8)
ax.set_title(f"Volatilidad anualizada (ventana rodante de {window} días)")
ax.set_ylabel("Volatilidad")
ax.legend(loc="upper right")
plt.tight_layout()

---
## 5. Correlación rodante

La correlación entre dos activos **no es constante** en el tiempo. La **correlación rodante** permite visualizar cómo evoluciona la relación de co-movimiento:

$$
\hat{\rho}_{ij,t}(n) = \frac{\sum_{k=0}^{n-1}(r_{i,t-k} - \bar{r}_{i,t})(r_{j,t-k} - \bar{r}_{j,t})}{\sqrt{\sum_{k=0}^{n-1}(r_{i,t-k} - \bar{r}_{i,t})^2 \cdot \sum_{k=0}^{n-1}(r_{j,t-k} - \bar{r}_{j,t})^2}}
$$

En períodos de crisis, las correlaciones tienden a **aumentar** (contagio financiero), reduciendo los beneficios de la diversificación justo cuando más se necesitan (Longin & Solnik, 2001).

In [ ]:
window = 20
rolling_corr = r["AAPL"].rolling(window).corr(r["MSFT"]).dropna()

fig, ax = plt.subplots(figsize=(12, 5))
rolling_corr.plot(ax=ax, color="steelblue")
ax.axhline(r["AAPL"].corr(r["MSFT"]), color="red", linestyle="--",
           label=f"Correlación global: {r['AAPL'].corr(r['MSFT']):.3f}")
ax.set_title(f"Correlación rodante AAPL vs MSFT (ventana {window} días)")
ax.set_ylabel("Correlación")
ax.legend()
plt.tight_layout()

---
## 6. Distribución de rendimientos

### Histogramas con ajuste normal

Comparamos la distribución empírica de cada activo con la distribución normal teórica. Las desviaciones (colas pesadas, asimetría) son los **hechos estilizados** de los rendimientos financieros (Cont, 2001).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, r.columns):
    sns.histplot(r[col], bins=40, stat="density", kde=True, ax=ax, alpha=0.6)
    # Overlay normal fit
    x = np.linspace(r[col].min(), r[col].max(), 200)
    ax.plot(x, stats.norm.pdf(x, r[col].mean(), r[col].std()),
            "r--", linewidth=2, label="Normal teórica")
    ax.set_title(f"Distribución — {col}")
    ax.set_xlabel("Rendimiento log")
    ax.legend(fontsize=9)
plt.suptitle("Distribución de rendimientos logarítmicos vs. normal", fontsize=14)
plt.tight_layout()

---
## 7. Relaciones bivariadas entre activos

### 7.1 Matriz de dispersión (pairplot)

La **matriz de dispersión** muestra todas las relaciones bivariadas entre activos. En la diagonal se muestra la distribución marginal (KDE); fuera de la diagonal, los diagramas de dispersión revelan la estructura de dependencia.

In [ ]:
sns.pairplot(r, diag_kind="kde", plot_kws={"alpha": 0.3, "s": 15})
plt.suptitle("Matriz de dispersión de rendimientos logarítmicos", y=1.01)
plt.tight_layout()

### 7.2 Distribuciones conjuntas (jointplot)

El **jointplot** combina el diagrama de dispersión con las distribuciones marginales y una estimación de densidad por kernel (KDE) que revela la estructura de la distribución conjunta.

In [ ]:
pairs = [("AA", "MSFT"), ("AAPL", "MSFT"), ("^GSPC", "MSFT")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (x_col, y_col) in zip(axes, pairs):
    ax.scatter(r[x_col], r[y_col], alpha=0.3, s=15, color="steelblue")
    # Regression line
    z = np.polyfit(r[x_col], r[y_col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(r[x_col].min(), r[x_col].max(), 100)
    ax.plot(x_line, p(x_line), "r-", linewidth=2,
            label=f"β = {z[0]:.3f}")
    rho = r[x_col].corr(r[y_col])
    ax.set_title(f"{x_col} vs {y_col}  (ρ = {rho:.3f})")
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.legend()
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.axvline(0, color="gray", linewidth=0.5, linestyle="--")
plt.suptitle("Dispersión y regresión lineal entre pares de activos", fontsize=14)
plt.tight_layout()

### 7.3 Regresión lineal entre rendimientos

La pendiente de la regresión de los rendimientos de un activo sobre el mercado (S&P 500) estima el **coeficiente beta** ($\beta$) del modelo CAPM:

$$
r_i = \alpha + \beta \, r_m + \varepsilon
$$

donde $\beta = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)}$ mide la **sensibilidad** del activo al riesgo de mercado (Sharpe, 1964).

- $\beta > 1$: el activo es más volátil que el mercado.
- $\beta < 1$: menos volátil.
- $\beta \approx 1$: se mueve con el mercado.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
stocks = ["AA", "AAPL", "MSFT"]
for ax, stock in zip(axes, stocks):
    sns.regplot(x=r["^GSPC"], y=r[stock], ax=ax,
                scatter_kws={"alpha": 0.3, "s": 15}, color="steelblue",
                line_kws={"color": "red"})
    beta = r[stock].cov(r["^GSPC"]) / r["^GSPC"].var()
    ax.set_title(f"{stock} vs S&P 500  (β = {beta:.3f})")
    ax.set_xlabel("^GSPC")
    ax.set_ylabel(stock)
plt.suptitle("Regresión de rendimientos sobre el mercado (estimación de β)", fontsize=14)
plt.tight_layout()

---
## 8. Rendimiento acumulado

El **rendimiento acumulado** muestra el valor de $1 USD invertido al inicio del período para cada activo:

$$
\frac{S_T}{S_0} = \exp\!\left(\sum_{t=1}^{T} r_t\right)
$$

In [ ]:
cum_r = np.exp(r.cumsum())

fig, ax = plt.subplots()
cum_r.plot(ax=ax, alpha=0.8)
ax.axhline(1, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Rendimiento acumulado (base = 1 USD)")
ax.set_ylabel("Valor de la inversión")
ax.legend()
plt.tight_layout()

---

## 9. Referencias bibliográficas

- **Cont, R.** (2001). Empirical Properties of Asset Returns: Stylized Facts and Statistical Issues. *Quantitative Finance*, 1(2), 223–236.
- **Engle, R. F.** (1982). Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation. *Econometrica*, 50(4), 987–1007.
- **Fama, E. F.** (1970). Efficient Capital Markets: A Review of Theory and Empirical Work. *The Journal of Finance*, 25(2), 383–417.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 22: Estimating Volatilities and Correlations.
- **Longin, F. & Solnik, B.** (2001). Extreme Correlation of International Equity Markets. *The Journal of Finance*, 56(2), 649–676.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8: Mean-Variance Portfolio Theory.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **McNeil, A. J., Frey, R. & Embrechts, P.** (2015). *Quantitative Risk Management* (2nd ed.). Princeton University Press. — Cap. 3: Empirical Properties of Financial Data.
- **Sharpe, W. F.** (1964). Capital Asset Prices: A Theory of Market Equilibrium under Conditions of Risk. *The Journal of Finance*, 19(3), 425–442.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. — Cap. 1–2.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos: productos derivados y decisiones económicas bajo incertidumbre* (2a ed.). Cengage Learning.